# Stage 7 — Prune Low-Likelihood Branches

Drops candidate diagnoses from **Evidence Scoring** (Stage 6) whose score falls below
`PRUNE_SCORE_THRESHOLD` (default 40 — see the Stage 6 scoring guide: 0-20 minimal, 21-40 weak,
41-60 moderate, 61-80 strong, 81-100 very strong).

This is a **deterministic cutoff**, not an LLM call — no LLM connection needed for this stage.

**Input:** evidence scoring per-patient files (Stage 6)
**Output:** one JSON file per patient in `data/stage_07_pruned_branches/` + an index


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

import pandas as pd
from pipeline import (
    PRUNE_SCORE_THRESHOLD,
    load_evidence_scoring_results,
    print_pipeline_banner,
    prune_low_likelihood_branches,
    save_pruning_results,
)

print_pipeline_banner()
print(f"Prune score threshold: {PRUNE_SCORE_THRESHOLD}")

scoring_df = load_evidence_scoring_results()
print(f"Admissions to prune: {len(scoring_df)}")

## Prune Candidates Below Threshold

In [ ]:
pruned = []

for _, row in scoring_df.iterrows():
    hadm_id = str(row["hadm_id"])
    patient_id = str(row["patient_id"])
    result = prune_low_likelihood_branches(
        evidence_scoring=row["evidence_scoring"],
        admission_id=hadm_id,
        patient_id=patient_id,
    )
    print(
        f"Pruned patient={patient_id} hadm_id={hadm_id}: "
        f"kept {result['n_kept']}/{result['n_input_candidates']}"
    )
    pruned.append({
        "patient_id": patient_id,
        "admission_id": row["admission_id"],
        "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]),
        "primary_icd_code": row["primary_icd_code"],
        "primary_dx_title": row["primary_dx_title"],
        "ground_truth_icd10": row["ground_truth_icd10"],
        "ground_truth_dx_titles": row["ground_truth_dx_titles"],
        "branch_pruning": result,
        "n_kept": result["n_kept"],
        "n_pruned": result["n_pruned"],
    })

results_df = pd.DataFrame(pruned)
results_df[["patient_id", "admission_id", "n_kept", "n_pruned"]]

## Save Results

In [ ]:
out = save_pruning_results(results_df)
print(f"Saved → {out}")

n_zero_kept = (results_df["n_kept"] == 0).sum()
if n_zero_kept:
    print(
        f"WARNING: {n_zero_kept} patient(s) had zero candidates survive pruning "
        f"(all scores < {PRUNE_SCORE_THRESHOLD})."
    )